In [ ]:
import sys
import logging
import html2text
from playwright.async_api import async_playwright
import nest_asyncio
import asyncio
from typing import List
import os
from requests_html import AsyncHTMLSession
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import StorageContext, load_index_from_storage, SimpleDirectoryReader, VectorStoreIndex, Settings
from dotenv import load_dotenv
load_dotenv()
nest_asyncio.apply()

c:\Users\Abhinav\Desktop\LLM_portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import openai
print("openai version:", openai.__version__)
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

openai version: 2.20.0


In [3]:
chat_llm = AzureOpenAI(deployment_name = os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'),
                       model = str(os.getenv('AZURE_OPENAI_MODEL')),
                       api_key = os.getenv('AZURE_OPENAI_API_KEY'),
                       api_version = os.getenv('AD_OPENAI_API_VERSION'),
                                        azure_endpoint = os.getenv('AD_OPENAI_ENDPOINT'))
                                        
embedding_llm = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Settings.llm = chat_llm
Settings.embed_model = embedding_llm

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/co

INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 496.50it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb9

In [ ]:
async def download_and_save_in_markdown(
                                url: str, 
                                dir_path: str,
                                session: AsyncHTMLSession
                                ) -> None:
    """Download the HTML content asynchronously from the web page and save as markdown."""
    # Extract a filename from the URL
    if url.endswith("/"):
        url = url[:-1]

    filename = url.split("/")[-1] + ".md"
    print(f"Downloading {url} into {filename}...")

    nest_asyncio.apply()

    response = await session.get(url, timeout=30) # type: ignore

    # Render the page, which will execute JavaScript
    await response.html.arender(timeout=20)

    # Convert the rendered HTML content to markdown
    h = html2text.HTML2Text()
    markdown_content = h.handle(response.html.raw_html.decode("utf-8"))

    # Write the markdown content to a file
    filename = os.path.join(dir_path, filename)
    if not os.path.exists(filename):
        with open(filename, "w", encoding="utf-8") as f:
            f.write(markdown_content)


async def download(pages: List[str]) -> str:
    """Download the HTML content from the pages and save them as markdown files."""
    # Create the content/notion directory if it doesn't exist
    # base_dir = os.path.dirname(os.path.abspath(__file__))
    dir_path = "blogs"
    os.makedirs(dir_path, exist_ok=True)
    session = AsyncHTMLSession()    
    tasks = [download_and_save_in_markdown(page, dir_path, session) for page in pages]
    await asyncio.gather(*tasks)
    await session.close()
    return dir_path


PAGES = [
    "https://quickstarts.snowflake.com/guide/data_engineering_pipelines_with_snowpark_python",
    "https://quickstarts.snowflake.com/guide/cloud_native_data_engineering_with_matillion_and_snowflake",
    "https://quickstarts.snowflake.com/guide/data_engineering_with_apache_airflow",
    "https://quickstarts.snowflake.com/guide/getting_started_with_dataengineering_ml_using_snowpark_python",
    "https://quickstarts.snowflake.com/guide/data_engineering_with_snowpark_python_and_dbt"
]

asyncio.run(download(PAGES))

[INFO] Starting Chromium download.


INFO:pyppeteer.chromium_downloader:Starting Chromium download.
Starting Chromium download.


[INFO] Starting Chromium download.


INFO:pyppeteer.chromium_downloader:Starting Chromium download.
Starting Chromium download.


OSError: Chromium downloadable not found at https://storage.googleapis.com/chromium-browser-snapshots/Win_x64/1181205/chrome-win.zip: Received <?xml version='1.0' encoding='UTF-8'?><Error><Code>NoSuchKey</Code><Message>The specified key does not exist.</Message><Details>No such object: chromium-browser-snapshots/Win_x64/1181205/chrome-win.zip</Details></Error>.


...

INFO:pyppeteer.chromium_downloader:Starting Chromium download.
Starting Chromium download.


[INFO] Starting Chromium download.


INFO:pyppeteer.chromium_downloader:Starting Chromium download.
Starting Chromium download.


[INFO] Starting Chromium download.


INFO:pyppeteer.chromium_downloader:Starting Chromium download.
Starting Chromium download.


In [ ]:
def build_index(
                data_dir: str, 
                knowledge_base_dir: str
                ) -> None:
    """Build the vector index from the markdown files in the directory."""

    print("Building vector index...")
    documents = SimpleDirectoryReader(data_dir).load_data()

    # index = TreeIndex.from_documents(documents, service_context=service_context)
    index = VectorStoreIndex.from_documents(
                                                    documents, 
                                                    service_context=service_context, 
                                                    show_progress=True
                                                    )
    index.storage_context.persist(persist_dir=knowledge_base_dir)
    print("Done.")

In [ ]:
build_index('blogs/', 'kb/')

In [ ]:
def load_index(knowledge_base_dir: str) -> VectorStoreIndex:
    """Load the vector index from the directory."""
    print("Loading vector index...")
    storage_context = StorageContext.from_defaults(persist_dir=knowledge_base_dir)
    index = load_index_from_storage(storage_context=storage_context)
    query_engine = index.as_query_engine()
    print("Done.")
    return query_engine

In [ ]:
query_engine = load_index('kb/')

In [ ]:
def chat_inference(
                    query: str,
                    top_k: int = 5
                    ) -> None:
    """Run a query against the index and print the results."""
    results = query_engine.query(query)
    print(f"Query: {query}")
    print(f"Response: {results}")

In [ ]:
chat_inference("what data engineers are focused primarily ?")